## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


In [ ]:
"""
Visualization Notebook for T5 and BART
Heatmaps, UMAP Scatter Plots, Loss Curves
"""

### Optional Google Colab use
Google Drive mounting is not required. In Colab, clone or upload the complete repository and run the notebook from the repository root. External model weights, when needed, must be placed under `external_materials/model_weights/`.


In [ ]:
import os
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login(key=os.environ["WANDB_API_KEY"]) if os.getenv("WANDB_API_KEY") else None

In [ ]:
# ================================================================
# INSTALLS
# ================================================================
!pip install -q transformers umap-learn seaborn bitsandbytes

import os
import math
import random
import numpy as np
import pandas as pd
import torch
import string
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
import umap.umap_ as umap

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM
)

import nltk
from nltk.tag import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [ ]:
# ================================================================
# RUNTIME ENVIRONMENT
# ================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ================================================================
# NLTK
# ================================================================
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# ================================================================
# TEXT CLEAN FUNCTION (identical to training)
# ================================================================
def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    mapping = {"J": wordnet.ADJ, "N": wordnet.NOUN,
               "V": wordnet.VERB, "R": wordnet.ADV}
    return mapping.get(tag, wordnet.NOUN)

def clean_text(text):
    stop_words = set(stopwords.words("english"))
    lem = WordNetLemmatizer()
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = word_tokenize(text)
    words = [w for w in words if w not in stop_words]
    words = [lem.lemmatize(w, get_wordnet_pos(w)) for w in words]
    return " ".join(words)

In [ ]:
# ================================================================
# LOAD DATASET (same as training)
# ================================================================
DATA_PATH = "data/external/sentences_dataset_45269.csv"

df = pd.read_csv(DATA_PATH)

#df["clean_sentence"] = df["sentence"].apply(clean_text)

df = df.drop_duplicates(subset="sentence", keep="first")

train_data, temp_data = train_test_split(
    df, test_size=0.3, stratify=df["polarity"], random_state=SEED
)
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, stratify=temp_data["polarity"], random_state=SEED
)

test_sentences = test_data["sentence"].tolist()
test_labels = test_data["polarity"].tolist()

LABEL_NAMES = ["Negative", "Neutral", "Positive"]

In [ ]:
# ---------------------------------------------------------------
# 1) BART INFERENCE
# ---------------------------------------------------------------
BART_MODEL = "external_materials/model_weights/facebook_bart-large/saved_model"
BART_TOKENIZER = "external_materials/model_weights/facebook_bart-large/saved_tokenizer"

print("\nLoading BART model...")
bart_tokenizer = AutoTokenizer.from_pretrained(BART_TOKENIZER)
bart_model = AutoModelForSequenceClassification.from_pretrained(BART_MODEL).to(DEVICE)
bart_model.eval()

def bart_predict(sentences, batch_size=16):
    preds, embs = [], []

    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i+batch_size]

            enc = bart_tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            ).to(DEVICE)

            # ---------------------------
            # 1) CLASSIFICATION PREDICTION
            # ---------------------------
            outputs = bart_model(**enc)
            logits = outputs.logits
            batch_preds = torch.argmax(logits, dim=1).cpu().tolist()
            preds.extend(batch_preds)

            # ---------------------------------------
            # 2) EXTRACT EMBEDDINGS FROM ENCODER ONLY
            # ---------------------------------------
            encoder_outputs = bart_model.model.encoder(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                output_hidden_states=False,
                return_dict=True
            )

            # use last hidden state (B, L, H) → take CLS-like first token
            encoder_hidden = encoder_outputs.last_hidden_state[:, 0, :]
            embs.append(encoder_hidden.cpu().numpy())

    return np.array(preds), np.vstack(embs)

bart_preds, bart_embs = bart_predict(test_sentences)


In [ ]:
# ---------------------------------------------------------------
# 2) T5 INFERENCE (seq2seq → generate → label mapping)
# ---------------------------------------------------------------
T5_MODEL = "external_materials/model_weights/t5-large/saved_model"
T5_TOKENIZER = "external_materials/model_weights/t5-large/saved_tokenizer"

print("\nLoading T5 model...")
t5_tokenizer = AutoTokenizer.from_pretrained(T5_TOKENIZER)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL).to(DEVICE)
t5_model.eval()

inv_map = {"negative": 0, "neutral": 1, "positive": 2}

def t5_predict(sentences, batch_size=16):
    preds, embs = [], []

    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch = ["classify sentiment: " + s for s in sentences[i:i+batch_size]]

            enc = t5_tokenizer(batch, padding=True, truncation=True,
                               max_length=256, return_tensors="pt").to(DEVICE)

            outputs = t5_model.generate(
                enc["input_ids"],
                max_new_tokens=5
            )
            decoded = [t5_tokenizer.decode(o, skip_special_tokens=True).strip().lower()
                       for o in outputs]

            batch_preds = []
            for text in decoded:
                if "negative" in text:
                    batch_preds.append(0)
                elif "positive" in text:
                    batch_preds.append(2)
                else:
                    batch_preds.append(1)

            preds.extend(batch_preds)

            # ------------------------------
            # embeddings from encoder output
            # ------------------------------
            enc_outputs = t5_model.encoder(**enc, output_hidden_states=True)
            hidden = enc_outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embs.append(hidden)

    return np.array(preds), np.vstack(embs)

t5_preds, t5_embs = t5_predict(test_sentences)

In [ ]:
def save_classification_report(model_name, y_true, y_pred, class_names, save_path):
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=3
    )

    fig, ax = plt.subplots(figsize=(8.5, 6))
    ax.axis("off")

    ax.text(
        0.01, 0.99,
        f"Classification Report — {model_name}\n\n{report}",
        fontsize=10,
        va="top",
        family="monospace"
    )

    fig.tight_layout()
    fig.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.close(fig)


report_path = f"outputs/figures/fig_comparable/T5_classification_report.pdf"
save_classification_report("T5", test_labels, t5_preds, LABEL_NAMES, report_path)
print(f"Saved classification report for T5 -> {report_path}")

report_path2 = f"outputs/figures/fig_comparable/Bart_classification_report.pdf"
save_classification_report("Bart", test_labels, bart_preds, LABEL_NAMES, report_path2)
print(f"Saved classification report for Bart -> {report_path2}")

In [ ]:
# ---------------------------------------------------------------
# HEATMAPS FOR T5 + BART
# ---------------------------------------------------------------
sns.set(style="whitegrid", font_scale=1.0)

cms = {
    "T5-large": confusion_matrix(test_labels, t5_preds),
    "BART-large": confusion_matrix(test_labels, bart_preds)
}

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for ax, (name, cm) in zip(axes, cms.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, cbar=False, ax=ax)

    ax.set_title(f"{name}")
    ax.set_xlabel("Predicted Labels")
    ax.set_ylabel("True Labels")

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig(
    'outputs/figures/fig_comparable/T5_BART_heatmaps_2_per_row.pdf',
    format='pdf',
    bbox_inches='tight',
    dpi=300
)

In [ ]:
# ---------------------------------------------------------------
# UMAP SCATTER
# ---------------------------------------------------------------
sns.set(style="whitegrid", font_scale=1.0)
models_embs = {"T5-large": t5_embs, "BART-large": bart_embs}

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes = axes.flatten()

for ax, (name, emb) in zip(axes, models_embs.items()):
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED)
    emb2d = reducer.fit_transform(emb)

    df_plot = pd.DataFrame({
        "x": emb2d[:, 0],
        "y": emb2d[:, 1],
        "label": [LABEL_NAMES[i] for i in test_labels]
    })

    palette = {
        "Negative": "tab:red",
        "Neutral": "tab:blue",
        "Positive": "tab:green"
    }

    sns.scatterplot(data=df_plot, x="x", y="y", hue="label",
                    s=20, alpha=0.7, palette=palette, ax=ax)

    ax.set_title(f"{name} Embeddings")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(title="Sentiment", loc="best", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig(
    'outputs/figures/fig_comparable/T5_BART_umap_scatter_2_per_row.pdf',
    format='pdf',
    bbox_inches='tight',
    dpi=300
)

In [ ]:
# ---------------------------------------------------------------
# LOSS CURVES
# ---------------------------------------------------------------
loss_files = {
    "T5-large": "data/processed/logs/loss_t5-large.csv",
    "BART-large": "data/processed/logs/loss_facebook_bart-large.csv"
}

train_color = "#1f77b4"   # Blue
val_color   = "#ff7f0e"   # Orange

sns.set(style="whitegrid", font_scale=1.0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for ax, (name, path) in zip(axes, loss_files.items()):
    df = pd.read_csv(path)

    sns.lineplot(ax=ax, data=df, x="epoch", y="train_loss",
                 marker="o", markersize=7, linewidth=2, color=train_color, label="Train")

    if "val_loss" in df.columns:
        sns.lineplot(ax=ax, data=df, x="epoch", y="val_loss",
                     marker="X", markersize=7, linewidth=2, linestyle="--", color=val_color, label="Validation")

    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylim(0.0, 1.0)
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

    ax.grid(True)
    ax.legend(fontsize=10, loc="best")

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig(
        "outputs/figures/fig_comparable/T5_BART_loss_curves.pdf",
        format="pdf",
        bbox_inches="tight",
        dpi=300
    )

Δημιουργία Master Results File.
Παρακάτω δημιουργείται το αρχείο για πρώτη φορά.

In [ ]:
# ---------------------------------------------------------------
# SAVE TEST SET PREDICTIONS TO CSV
# ---------------------------------------------------------------

results_df = pd.DataFrame({
    "citation": test_sentences,
    "ground_truth": test_labels,
    "BART_pred": bart_preds,
    "T5_pred": t5_preds
})

SAVE_PATH = (
    str(PROCESSED_DATA_DIR / "Master_results.csv")
)

results_df.to_csv(SAVE_PATH, index=False, encoding="utf-8")

print(f"Saved test predictions CSV -> {SAVE_PATH}")

In [ ]:
results_df

END